[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/prefect-certified/notebooks/day-09-schedules-automations.ipynb#scrollTo=b1c2d3e4)

---
# Day 9 · Schedules, Automations, and Event-Driven Triggers
**certified-journeys / prefect-certified** &nbsp;|&nbsp; Practice

> **Goal for today:** Attach `CronSchedule` and `IntervalSchedule` to deployments, build automations that react to flow state events, create webhook-triggered runs, and understand how to pause/resume schedules without losing runs.

---
## Scheduling and automation overview

Prefect separates **when** to run (schedules) from **what causes a run** (automations):

| Mechanism | Trigger | Use case |
|---|---|---|
| `CronSchedule` | Wall-clock cron expression | Daily/weekly jobs at a fixed time |
| `IntervalSchedule` | Fixed time delta from an anchor | Every N hours/minutes with drift protection |
| `RRuleSchedule` | iCal RRULE | Complex recurrences (every 3rd Monday) |
| Automation — state trigger | Another flow enters a state | Chain flows, alert on failure |
| Automation — webhook | External HTTP POST | Run on Slack slash command, GitHub push, etc. |
| Automation — metric | Metric threshold crossed | Scale up workers on backlog growth |

**Key insight:** Schedules live on deployments. Automations live on the server. Both create runs through the same API — a run is a run regardless of how it was triggered.

In [ ]:
%pip install -q "prefect>=2.14" croniter

---
## Step 1 · CronSchedule — weekday runs at 07:00 UTC

A `CronSchedule` uses standard cron syntax. The server creates runs at each due time and places them in the work pool queue.

**Cron quick reference:**

```
┌─ minute    (0–59)
│ ┌─ hour    (0–23)
│ │ ┌─ day   (1–31)
│ │ │ ┌─ month (1–12)
│ │ │ │ ┌─ weekday (0=Sun … 6=Sat, or MON–FRI)
│ │ │ │ │
0 7 * * MON-FRI     ← every weekday at 07:00
0 9 * * 1           ← every Monday at 09:00
*/15 * * * *        ← every 15 minutes
0 0 1 * *           ← first day of every month
```

**CronSchedule parameters:**

| Parameter | Type | Description |
|---|---|---|
| `cron` | str | Cron expression (5 fields) |
| `timezone` | str | IANA tz name, e.g. `"America/New_York"` |
| `day_or` | bool | `True` = day-of-month OR day-of-week (cron default) |

In [ ]:
import os
import prefect
from prefect.server.schemas.schedules import CronSchedule
from croniter import croniter
from datetime import datetime, timezone

os.environ["PREFECT_API_URL"] = ""
os.environ["PREFECT_SERVER_ANALYTICS_ENABLED"] = "false"

print(f"Prefect version: {prefect.__version__}")

# Define a CronSchedule: every weekday (Mon–Fri) at 07:00 UTC
weekday_schedule = CronSchedule(
    cron="0 7 * * MON-FRI",
    timezone="UTC",
)

print("\nCronSchedule definition:")
print(f"  cron:     {weekday_schedule.cron}")
print(f"  timezone: {weekday_schedule.timezone}")

# Use croniter to preview the next 7 scheduled run times
cron = croniter("0 7 * * MON-FRI", start_time=datetime(2024, 1, 15, 0, 0, tzinfo=timezone.utc))

print("\nNext 7 scheduled runs:")
for _ in range(7):
    next_run = cron.get_next(datetime)
    day_name = next_run.strftime("%A")
    print(f"  {next_run.strftime('%Y-%m-%d %H:%M UTC')}  ({day_name})")

# Show how to attach a schedule to a deployment via the Python API
print("\nHow Prefect stores the schedule (JSON):")
import json
print(json.dumps({"cron": "0 7 * * MON-FRI", "timezone": "UTC", "day_or": True}, indent=2))

**What just happened?**
- `CronSchedule(cron="0 7 * * MON-FRI", timezone="UTC")` defines exact wall-clock run times
- **Always specify a timezone** — `UTC` avoids DST surprises; `America/New_York` shifts with DST automatically
- The server pre-generates runs up to a configurable horizon (default: 100 upcoming runs) — so runs survive short outages
- `croniter` is a useful local library for validating cron expressions before deploying

---
## Step 2 · IntervalSchedule — every 6 hours with an anchor date

An `IntervalSchedule` fires every fixed delta from an **anchor date**.
The anchor ensures the schedule is deterministic — you always know exactly when runs will occur.

**Why anchor date matters:**

Without an anchor, "every 6 hours" depends on when the schedule was created.  
With an anchor of `2024-01-01T00:00:00Z`, runs are always at `00:00`, `06:00`, `12:00`, `18:00` — forever.

| Parameter | Type | Description |
|---|---|---|
| `interval` | timedelta | Gap between consecutive runs |
| `anchor_date` | datetime | Reference point the interval is measured from |
| `timezone` | str | IANA tz name |

**IntervalSchedule vs CronSchedule:**

| | CronSchedule | IntervalSchedule |
|---|---|---|
| Best for | Time-of-day precision (07:00 on weekdays) | Fixed cadence (every 6 h) |
| DST handling | Automatic (follow wall clock) | Fixed UTC interval (ignores DST) |
| Expression | Cron string | `timedelta` object |

In [ ]:
from prefect.server.schemas.schedules import IntervalSchedule
from datetime import timedelta, datetime, timezone

# IntervalSchedule: every 6 hours, anchored to 2024-01-01 00:00:00 UTC
six_hour_schedule = IntervalSchedule(
    interval=timedelta(hours=6),
    anchor_date=datetime(2024, 1, 1, 0, 0, 0, tzinfo=timezone.utc),
    timezone="UTC",
)

print("IntervalSchedule definition:")
print(f"  interval:    {six_hour_schedule.interval}")
print(f"  anchor_date: {six_hour_schedule.anchor_date}")
print(f"  timezone:    {six_hour_schedule.timezone}")

# Compute the next N runs manually from the anchor
def get_interval_runs(anchor: datetime, interval: timedelta, start_after: datetime, n: int):
    """Generate the next n scheduled runs after start_after."""
    # Find the first run >= start_after
    elapsed = (start_after - anchor).total_seconds()
    intervals_elapsed = int(elapsed / interval.total_seconds())
    current = anchor + interval * (intervals_elapsed + 1)
    runs = []
    for _ in range(n):
        runs.append(current)
        current += interval
    return runs

start = datetime(2024, 1, 15, 9, 30, tzinfo=timezone.utc)
upcoming = get_interval_runs(
    anchor=datetime(2024, 1, 1, 0, 0, 0, tzinfo=timezone.utc),
    interval=timedelta(hours=6),
    start_after=start,
    n=8,
)

print(f"\nNext 8 runs after {start.strftime('%Y-%m-%d %H:%M UTC')}:")
for run_time in upcoming:
    print(f"  {run_time.strftime('%Y-%m-%d %H:%M UTC')}")

print("\n--- CLI to set a schedule on an existing deployment ---")
print("prefect deployment set-schedule 'etl-pipeline/etl-daily' \\ ")
print("  --interval 21600  # 6 hours in seconds")

**What just happened?**
- `interval=timedelta(hours=6)` + `anchor_date` produces perfectly aligned runs: `00:00`, `06:00`, `12:00`, `18:00` UTC
- The anchor is the **mathematical origin** — Prefect computes run times as `anchor + n * interval`; no drift accumulates
- **Use `IntervalSchedule` for data pipelines where freshness SLA matters** (e.g. refresh every 6 h) and `CronSchedule` when business hours matter (e.g. 07:00 weekdays)

---
## Step 3 · Attach a schedule to a deployment

Schedules are attached to deployments, not flows. The same flow can have multiple deployments,
each with a different schedule — e.g. a `fast` deployment every 15 min and a `daily` deployment once a day.

```bash
# CLI: create a deployment with a cron schedule in one command
prefect deploy etl_pipeline.py:etl_pipeline \
  --name etl-weekday-07 \
  --pool my-pool \
  --cron "0 7 * * MON-FRI" \
  --timezone UTC
```

Via Python, pass the schedule object when building the deployment.

In [ ]:
import asyncio
import json
from datetime import timedelta, datetime, timezone
from prefect import flow, task, get_run_logger
from prefect.server.schemas.schedules import CronSchedule, IntervalSchedule
from prefect.client.orchestration import get_client
from prefect.client.schemas.actions import WorkPoolCreate, DeploymentCreate

@task
def extract_records(source: str, limit: int) -> list[dict]:
    logger = get_run_logger()
    logger.info(f"Extracting {limit} records from '{source}'")
    return [{"id": i, "source": source, "value": i * 2.5} for i in range(1, limit + 1)]

@flow(name="scheduled-etl", log_prints=True)
def scheduled_etl(source: str = "api", limit: int = 100) -> dict:
    records = extract_records(source=source, limit=limit)
    total = sum(r["value"] for r in records)
    print(f"Processed {len(records)} records, total={total:.2f}")
    return {"count": len(records), "total": round(total, 2)}


# Register two deployments with different schedules
async def register_scheduled_deployments():
    async with get_client() as client:
        # Create the work pool
        try:
            await client.create_work_pool(WorkPoolCreate(name="schedule-pool", type="process"))
            print("Work pool 'schedule-pool' created")
        except Exception:
            print("Work pool 'schedule-pool' already exists")

        flow_id = await client.create_flow_from_name("scheduled-etl")

        # Deployment 1: CronSchedule — weekdays at 07:00 UTC
        dep1 = await client.create_deployment(
            DeploymentCreate(
                name="etl-weekday-morning",
                flow_id=flow_id,
                entrypoint="scheduled_etl:scheduled_etl",
                parameters={"source": "production-api", "limit": 1000},
                work_pool_name="schedule-pool",
                schedule=CronSchedule(cron="0 7 * * MON-FRI", timezone="UTC"),
                tags=["etl", "cron", "weekday"],
            )
        )
        print(f"\nDeployment 1 registered: etl-weekday-morning (id={dep1})")
        print("  Schedule: CronSchedule — 0 7 * * MON-FRI (UTC)")

        # Deployment 2: IntervalSchedule — every 6 hours
        dep2 = await client.create_deployment(
            DeploymentCreate(
                name="etl-6h-refresh",
                flow_id=flow_id,
                entrypoint="scheduled_etl:scheduled_etl",
                parameters={"source": "streaming-api", "limit": 250},
                work_pool_name="schedule-pool",
                schedule=IntervalSchedule(
                    interval=timedelta(hours=6),
                    anchor_date=datetime(2024, 1, 1, 0, 0, 0, tzinfo=timezone.utc),
                    timezone="UTC",
                ),
                tags=["etl", "interval", "6h"],
            )
        )
        print(f"Deployment 2 registered: etl-6h-refresh     (id={dep2})")
        print("  Schedule: IntervalSchedule — every 6 hours from 2024-01-01 00:00 UTC")

        return dep1, dep2

d1, d2 = asyncio.run(register_scheduled_deployments())
print("\nBoth deployments are visible in the UI under the 'scheduled-etl' flow")

**What just happened?**
- Two deployments share the same flow code but have **different schedules and parameters** — a common pattern for dev vs prod or high-frequency vs daily
- `schedule=CronSchedule(...)` is stored in the deployment record — the server's scheduler loop reads it
- **Upcoming runs are pre-generated** — Prefect creates `Scheduled` flow run records ahead of time so they survive brief server restarts
- In the UI: **Deployments → scheduled-etl** shows both deployments with their next run time

---
## Step 4 · Pause and resume a schedule

Pausing a schedule stops the server from creating new `Scheduled` runs. Runs already in the queue
are **not cancelled** — they run normally.

**What happens on resume:**

| Past runs during pause | Behaviour |
|---|---|
| Runs that were pre-generated | Already in queue — they run when a worker picks them up |
| Runs that were NOT pre-generated | **Missed** — Prefect does not backfill by default |
| `catchup_runs=True` setting | Server creates missed runs on resume (useful for ETL) |

**CLI to pause/resume:**
```bash
# Pause
prefect deployment pause 'scheduled-etl/etl-weekday-morning'

# Resume
prefect deployment resume 'scheduled-etl/etl-weekday-morning'
```

In [ ]:
import asyncio
from prefect.client.orchestration import get_client

async def demonstrate_pause_resume(deployment_id: str):
    async with get_client() as client:
        # Read current deployment state
        dep = await client.read_deployment(deployment_id)
        print(f"Deployment: {dep.name}")
        print(f"  is_schedule_active: {dep.is_schedule_active}")

        # Pause the schedule
        await client.set_deployment_paused_state(deployment_id, paused=True)
        dep = await client.read_deployment(deployment_id)
        print(f"\nAfter pause:")
        print(f"  is_schedule_active: {dep.is_schedule_active}  ← False = paused")
        print("  Server will not create new Scheduled runs for this deployment")
        print("  Runs already in the queue will still execute normally")

        # Resume the schedule
        await client.set_deployment_paused_state(deployment_id, paused=False)
        dep = await client.read_deployment(deployment_id)
        print(f"\nAfter resume:")
        print(f"  is_schedule_active: {dep.is_schedule_active}  ← True = active")
        print("  Server resumes creating Scheduled runs from the next due time")
        print("  Runs missed during pause are NOT backfilled (unless catchup_runs=True)")

if d1:
    asyncio.run(demonstrate_pause_resume(str(d1)))
else:
    print("No deployment id — run Step 3 first")

print("\nUI path: Deployments → etl-weekday-morning → ⏸ Pause / ▶ Resume button")

**What just happened?**
- `set_deployment_paused_state(paused=True)` flips `is_schedule_active` to `False` — the scheduler loop skips this deployment
- **No runs are lost** that were already pre-generated and sitting in the queue
- Setting `paused=False` re-activates the schedule from the **next upcoming cron/interval time** — not from where it left off
- For ETL pipelines with strong backfill requirements, set `catchup_runs=True` in the deployment so missed runs are created on resume

---
## Step 5 · Automations — event-driven triggers

Automations watch for **events** and take **actions** when matching events are detected.

**Automation structure:**

```
Automation
├── trigger   (what to watch for)
│   ├── Event: flow run state change
│   ├── Event: work queue health
│   ├── Metric: queue depth > N
│   └── Compound: A and B within T seconds
└── actions   (what to do)
    ├── Run deployment
    ├── Pause deployment
    ├── Cancel flow run
    ├── Send notification (Slack, email, PagerDuty)
    └── Call webhook
```

**Common patterns:**

| Pattern | Trigger | Action |
|---|---|---|
| Alert on failure | Flow enters `Failed` state | Send Slack message |
| Chain flows | Flow A enters `Completed` | Run deployment B |
| Self-healing | Flow enters `Failed` | Run cleanup deployment |
| Backlog guard | Queue depth > 100 | Scale up workers |
| Dead-man switch | No run in 2 h | Page on-call engineer |

In [ ]:
import json

# Construct the JSON body for an automation that triggers a deployment
# when another flow enters a Failed state.
# This is the exact payload the Prefect Cloud / server API accepts.

AUTOMATION_PAYLOAD = {
    "name": "On ETL Failure — Run Cleanup",
    "description": "When the scheduled-etl flow fails, automatically trigger the cleanup-flow deployment.",
    "enabled": True,
    "trigger": {
        "type": "event",
        # Watch for state-change events
        "match": {
            "prefect.resource.id": "prefect.flow-run.*"
        },
        # Only match this specific flow by name
        "match_related": {
            "prefect.resource.role": "flow",
            "prefect.resource.name": "scheduled-etl"
        },
        # The event we're waiting for
        "expect": ["prefect.flow-run.Failed"],
        # No time window needed — trigger immediately on event
        "posture": "Reactive",
        "threshold": 1,
        "within": 0,
    },
    "actions": [
        {
            "type": "run-deployment",
            # deployment_id would be the UUID of the cleanup deployment
            "deployment_id": "<cleanup-deployment-uuid>",
            "parameters": {
                # Pass context from the triggering run
                "failed_run_id": "{{ event.resource.id }}",
                "failed_at": "{{ event.occurred }}",
            },
            "job_variables": {}
        }
    ]
}

print("Automation payload (POST /api/automations):")
print(json.dumps(AUTOMATION_PAYLOAD, indent=2))

print("\n--- Create via Prefect Cloud UI ---")
print("  Automations → New Automation → Event trigger → Flow Run Failed")
print("  → Select flow 'scheduled-etl' → Action: Run Deployment → cleanup-flow")

**What just happened?**
- The `trigger.expect` field is a list of **event type strings** — Prefect emits these as flows change state
- `match_related` narrows the trigger to a specific flow by name — without it, *any* failed run would trigger the action
- `{{ event.resource.id }}` is a **Jinja2 template** — Prefect fills it with the triggering run's ID at action execution time
- **Reactive posture** = trigger fires as soon as the event is seen (vs. **Proactive** = fire after N events in a time window)

---
## Step 6 · Automations in Python — building and reasoning about triggers

While automations are typically created in the UI or via API, understanding their structure
lets you build automated provisioning scripts and test their logic locally.

Let's build an automation system that demonstrates chaining flows and failure alerting.

In [ ]:
from dataclasses import dataclass, field
from typing import Callable, Optional
from datetime import datetime, timezone

# Minimal automation engine — mirrors how Prefect evaluates automations

@dataclass
class PrefectEvent:
    event_type: str       # e.g. "prefect.flow-run.Failed"
    resource_id: str      # flow run id
    resource_name: str    # flow name
    occurred: datetime
    payload: dict = field(default_factory=dict)

@dataclass
class AutomationTrigger:
    event_types: list[str]          # events to match
    resource_name: Optional[str]    # filter by flow name (None = any)

    def matches(self, event: PrefectEvent) -> bool:
        type_match = event.event_type in self.event_types
        name_match = (self.resource_name is None) or (event.resource_name == self.resource_name)
        return type_match and name_match

@dataclass
class AutomationAction:
    name: str
    execute: Callable[[PrefectEvent], None]

@dataclass
class Automation:
    name: str
    trigger: AutomationTrigger
    actions: list[AutomationAction]
    enabled: bool = True
    fire_count: int = 0

    def evaluate(self, event: PrefectEvent):
        if not self.enabled:
            return
        if self.trigger.matches(event):
            print(f"  [Automation '{self.name}'] TRIGGERED by event: {event.event_type}")
            for action in self.actions:
                print(f"    → Executing action: {action.name}")
                action.execute(event)
            self.fire_count += 1


# Define two automations
def run_cleanup_deployment(event: PrefectEvent):
    print(f"      [Action] Triggering cleanup deployment for failed run: {event.resource_id[:8]}…")
    print(f"      [Action] Parameters: failed_run_id={event.resource_id[:8]}…, failed_at={event.occurred}")

def send_slack_alert(event: PrefectEvent):
    print(f"      [Action] Sending Slack alert: Flow '{event.resource_name}' failed at {event.occurred}")

automation_cleanup = Automation(
    name="On ETL Failure — Run Cleanup",
    trigger=AutomationTrigger(
        event_types=["prefect.flow-run.Failed"],
        resource_name="scheduled-etl",  # only watch this specific flow
    ),
    actions=[AutomationAction("Run cleanup deployment", run_cleanup_deployment)],
)

automation_alert = Automation(
    name="Alert on Any Flow Failure",
    trigger=AutomationTrigger(
        event_types=["prefect.flow-run.Failed"],
        resource_name=None,  # match any flow
    ),
    actions=[AutomationAction("Send Slack alert", send_slack_alert)],
)

automations = [automation_cleanup, automation_alert]


# Simulate the event bus — Prefect emits events like these on every state change
events = [
    PrefectEvent(
        event_type="prefect.flow-run.Completed",
        resource_id="run-aaa-111",
        resource_name="scheduled-etl",
        occurred=datetime(2024, 1, 15, 7, 5, tzinfo=timezone.utc),
    ),
    PrefectEvent(
        event_type="prefect.flow-run.Failed",
        resource_id="run-bbb-222",
        resource_name="scheduled-etl",
        occurred=datetime(2024, 1, 15, 13, 5, tzinfo=timezone.utc),
    ),
    PrefectEvent(
        event_type="prefect.flow-run.Failed",
        resource_id="run-ccc-333",
        resource_name="other-pipeline",  # different flow
        occurred=datetime(2024, 1, 15, 14, 0, tzinfo=timezone.utc),
    ),
]

print("=== Processing events through automation engine ===")
for event in events:
    print(f"\nEvent: {event.event_type} | flow={event.resource_name}")
    for automation in automations:
        automation.evaluate(event)

print("\n=== Automation fire counts ===")
for a in automations:
    print(f"  '{a.name}': fired {a.fire_count} time(s)")

**What just happened?**
- The `AutomationTrigger` with `resource_name="scheduled-etl"` only fires on *that* flow — `other-pipeline` failures pass through to the general alert but not the cleanup action
- `resource_name=None` matches *any* flow failure — useful for a catch-all Slack alert
- Prefect evaluates automations on every event in real time — latency is typically < 1 second from event to action execution
- **Actions receive the triggering event** — use template variables like `{{ event.resource.id }}` to pass context into the triggered deployment's parameters

---
## Step 7 · Webhook-triggered automations

Prefect Cloud webhooks let external systems trigger flow runs via HTTP POST.

**How it works:**

```
External System                    Prefect Cloud
─────────────                      ─────────────
GitHub push event   ──POST──►  /api/webhooks/{webhook_id}
Stripe payment      ──POST──►      │
Slack slash cmd     ──POST──►      ▼
                               Emits custom event
                                   │
                                   ▼
                               Automation evaluates
                                   │
                               Action: Run deployment
```

**Webhook template** — maps the HTTP request to a Prefect event:

```json
{
  "event": "my-org.data-upload",
  "resource": {
    "prefect.resource.id": "my-org.upload.{{ body.upload_id }}",
    "source": "{{ body.source }}",
    "row_count": "{{ body.row_count }}"
  }
}
```

In [ ]:
import json
from string import Template

# Demonstrate webhook template rendering — this is what Prefect does internally
# when it receives an HTTP POST to a webhook endpoint

WEBHOOK_TEMPLATE = {
    "event": "acme.data-upload.received",
    "resource": {
        "prefect.resource.id": "acme.upload.{upload_id}",
        "source": "{source}",
        "row_count": "{row_count}",
        "file_path": "{file_path}",
    }
}

def render_webhook_event(template: dict, payload: dict) -> dict:
    """Render the webhook template with values from the HTTP POST body."""
    rendered = json.dumps(template)
    for key, val in payload.items():
        rendered = rendered.replace(f"{{{key}}}", str(val))
    return json.loads(rendered)


# Simulate an incoming webhook HTTP POST body
INCOMING_PAYLOAD = {
    "upload_id": "u-20240115-abc123",
    "source": "s3://acme-data/uploads/2024/01/15/",
    "row_count": 84_523,
    "file_path": "s3://acme-data/uploads/2024/01/15/data.parquet",
}

print("Incoming webhook payload (HTTP POST body):")
print(json.dumps(INCOMING_PAYLOAD, indent=2))

rendered_event = render_webhook_event(WEBHOOK_TEMPLATE, INCOMING_PAYLOAD)

print("\nRendered Prefect event:")
print(json.dumps(rendered_event, indent=2))

# Automation that listens for this event and runs a deployment
WEBHOOK_AUTOMATION = {
    "name": "On Data Upload — Run Ingestion Pipeline",
    "trigger": {
        "type": "event",
        "expect": ["acme.data-upload.received"],
        "match": {"prefect.resource.id": "acme.upload.*"},
        "posture": "Reactive",
        "threshold": 1,
        "within": 0,
    },
    "actions": [
        {
            "type": "run-deployment",
            "deployment_id": "<ingestion-deployment-uuid>",
            "parameters": {
                # Template variables filled from the event resource fields
                "file_path":  "{{ event.resource.file_path }}",
                "row_count":  "{{ event.resource.row_count }}",
                "upload_id":  "{{ event.resource['prefect.resource.id'] }}",
            }
        }
    ]
}

print("\nAutomation that triggers on this event:")
print(json.dumps(WEBHOOK_AUTOMATION, indent=2))

**What just happened?**
- The webhook template renders the HTTP POST body into a **Prefect event** — every field in `resource` is queryable in automations
- `expect: ["acme.data-upload.received"]` is a custom event type — prefix with your org name to avoid collisions with Prefect's built-in event types
- `match: {"prefect.resource.id": "acme.upload.*"}` uses glob matching — `*` matches any upload ID
- **Automation parameters** use `{{ event.resource.field }}` templates — the ingestion pipeline receives the exact file path and row count from the upload event

---
## Step 8 · Putting it all together — scheduled + event-driven pipeline

Real pipelines combine schedules and automations:

```
CronSchedule (07:00 UTC) ──► extract_flow
                                 │
                   Completed ──► transform_flow  (automation: on extract Completed)
                                 │
                   Completed ──► load_flow        (automation: on transform Completed)
                                 │
                   Failed   ──► alert + cleanup   (automation: on load Failed)
```

This replaces complex orchestration glue code with declarative automation rules.

In [ ]:
from prefect import flow, task, get_run_logger
from prefect.testing.utilities import prefect_test_harness
from datetime import datetime, timezone
import time

# Simulate the three-stage pipeline that automations would chain

@task(retries=2, retry_delay_seconds=2)
def extract_stage(source: str, batch_size: int) -> list[dict]:
    logger = get_run_logger()
    logger.info(f"EXTRACT: pulling {batch_size} records from {source}")
    return [{"id": i, "raw": i * 1.1, "source": source} for i in range(1, batch_size + 1)]

@task
def transform_stage(records: list[dict], filter_threshold: float) -> list[dict]:
    logger = get_run_logger()
    filtered = [r for r in records if r["raw"] >= filter_threshold]
    transformed = [{**r, "processed": round(r["raw"] * 2.5, 4)} for r in filtered]
    logger.info(f"TRANSFORM: {len(records)} in → {len(transformed)} out (threshold={filter_threshold})")
    return transformed

@task
def load_stage(records: list[dict], destination: str) -> int:
    logger = get_run_logger()
    logger.info(f"LOAD: writing {len(records)} records to '{destination}'")
    # Production: insert into database / write Parquet to S3
    return len(records)

@flow(name="full-etl-pipeline", log_prints=True)
def full_etl_pipeline(
    source: str = "api-v1",
    destination: str = "warehouse.events",
    batch_size: int = 100,
    filter_threshold: float = 5.0,
) -> dict:
    """Three-stage ETL — designed to be triggered by schedule or automation."""
    raw = extract_stage(source=source, batch_size=batch_size)
    transformed = transform_stage(records=raw, filter_threshold=filter_threshold)
    loaded = load_stage(records=transformed, destination=destination)

    summary = {
        "source": source,
        "destination": destination,
        "extracted": len(raw),
        "transformed": len(transformed),
        "loaded": loaded,
        "drop_rate": round(1 - loaded / len(raw), 3),
        "run_at": datetime.now(timezone.utc).isoformat(),
    }
    print(f"Pipeline complete: {summary}")
    return summary


# Run with the test harness — identical to how a scheduled or webhook-triggered run would execute
print("=== Scheduled run (simulates 07:00 UTC CronSchedule trigger) ===")
with prefect_test_harness():
    result_scheduled = full_etl_pipeline(
        source="production-api",
        destination="warehouse.events",
        batch_size=20,
        filter_threshold=10.0,
    )
    print(f"\nScheduled run result:")
    for k, v in result_scheduled.items():
        print(f"  {k}: {v}")

print()
print("=== Webhook-triggered run (simulates external upload event) ===")
with prefect_test_harness():
    result_webhook = full_etl_pipeline(
        source="s3://acme-data/uploads/2024/01/15/data.parquet",
        destination="warehouse.uploads",
        batch_size=8,
        filter_threshold=3.0,
    )
    print(f"\nWebhook run result:")
    for k, v in result_webhook.items():
        print(f"  {k}: {v}")

**What just happened?**
- The same `full_etl_pipeline` flow handles both the scheduled morning run and the ad-hoc webhook-triggered run — **flow code is trigger-agnostic**
- Different `source` and `destination` parameters are injected by the automation — no code changes between runs
- `drop_rate` captures how many records were filtered out — a critical metric to track across runs for data quality monitoring
- In production: add `@flow(retries=2)` and attach an automation to send an alert when `drop_rate > 0.5` (more than half the records dropped)

---
## Challenge

You have a nightly report flow and a cleanup flow. Your task is to wire them together with schedules and automations.

In [ ]:
# Challenge: Schedules and Automations
#
# Given these flows:
from prefect import flow, task
from prefect.testing.utilities import prefect_test_harness
from prefect.server.schemas.schedules import CronSchedule, IntervalSchedule
from croniter import croniter
from datetime import datetime, timedelta, timezone
import json

@task
def generate_report(date: str, region: str) -> dict:
    # Simulate report generation
    import hashlib
    digest = hashlib.md5(f"{date}{region}".encode()).hexdigest()[:8]
    return {"date": date, "region": region, "records": len(digest) * 100, "digest": digest}

@flow(name="nightly-report", log_prints=True)
def nightly_report(date: str = "2024-01-15", region: str = "US") -> dict:
    report = generate_report(date=date, region=region)
    print(f"Report generated: {report}")
    return report

@flow(name="cleanup-flow", log_prints=True)
def cleanup_flow(target_date: str = "2024-01-15") -> str:
    print(f"Cleaning up temp files for {target_date}")
    return f"Cleaned: {target_date}"

# Task 1: Run nightly_report with prefect_test_harness() for two regions: "US" and "EU"
#         using date="2024-01-15"
# YOUR CODE HERE

# Task 2: Print which region had more records
# YOUR CODE HERE

# Task 3: Create a CronSchedule for nightly_report that runs every day at 23:00 UTC
#         Use croniter to print the next 5 scheduled run times
# YOUR CODE HERE

# Task 4: Create an IntervalSchedule with interval=timedelta(hours=12)
#         and anchor_date=datetime(2024, 1, 1, 0, 0, 0, tzinfo=timezone.utc)
#         Print the schedule object's interval and anchor_date
# YOUR CODE HERE

# Task 5: Build the JSON payload for an automation that:
#         - Triggers when "nightly-report" flow enters a Completed state
#         - Action: run a deployment named "cleanup-daily"
#         Print the payload as formatted JSON
# YOUR CODE HERE

---
## Day 9 key concepts recap

| Concept | What to remember |
|---|---|
| `CronSchedule` | Wall-clock trigger — use for business-hour jobs; always set a timezone |
| `IntervalSchedule` | Fixed-delta trigger — use anchor_date for deterministic run times |
| Schedule active state | `is_schedule_active=False` = paused; existing queued runs still execute |
| Pause/resume | No backfill on resume (unless `catchup_runs=True`); next run from next due time |
| Automation trigger | Reactive (immediate) or Proactive (N events in T seconds) |
| Automation action | Run deployment, cancel run, send notification, call webhook |
| Event template vars | `{{ event.resource.id }}` fills from triggering event at action time |
| Webhook | External HTTP POST → custom event → automation → deployment run |
| Posture: Reactive | Trigger fires as soon as matching event is seen |
| Posture: Proactive | Trigger fires if event is NOT seen within a time window (dead-man switch) |

> **Tip:** Automations are the glue of event-driven pipelines — use them to chain flows, send alerts, or cancel stuck runs without any custom polling code.

---
## What's next
**Day 10** → Prefect observability: querying run history with the Python client, building custom dashboards with the events API, and setting up Prometheus metrics from a self-hosted Prefect server.

Mark Day 9 complete in your [tracker](../index.html).